# Question Answering with LangChain, OpenAI, and MultiQuery Retriever


<br>

## Intro

In this lab you'll build a Retrieval-Augmented Generation (RAG) chatbot backed by Elasticsearch and OpenAI, using LangChain's `MultiQueryRetriever` to improve retrieval quality.

**Why use a `MultiQueryRetriever`?** A single user question, phrased one way, might miss documents that use different wording for the same idea. [`MultiQueryRetriever`](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) asks an LLM to rephrase the original question into several alternative versions, runs a separate search for each version, and merges the results — giving you a broader, more relevant set of documents than a single search would.

Here's how the workflow will look like:

- **Setup (once):** split a sample dataset of documents into passages (chunks), turn each passage into an embedding, and store them in a vector database (in this case, we'll use Elasticsearch).
- **At question time:** the `MultiQueryRetriever` generates a few variations of the question, searches Elasticsearch with each one, and combines the retrieved passages. Those passages are then passed to an LLM, which uses them as context to write the final answer.


And here's what to expect from this lab:

1. **Complete a guided demo** — You'll be given the overall structure and process to follow: connecting to Elasticsearch and OpenAI, loading and chunking a sample set of workplace documents, embedding and indexing them into a vector store, and wiring up a `MultiQueryRetriever` chain that expands a question into several variations before retrieving and answering. Some steps are already implemented for you, and you'll need to complete the rest (e.g. filling in missing parameters and logic) to get it fully working.
2. **Replicate it yourself with variations** — Then, you'll create **at least two new iterations** of the question-answering step (new questions and/or settings) to explore how `MultiQueryRetriever` behaves.

By the end of this lab, you'll understand how multi-query retrieval works and be able to apply it to your own RAG pipelines.

<br>

## Install and import dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [1]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python executable:
c:\Users\aysea\AppData\Local\Programs\Python\Python311\python.exe

Python version:
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [2]:
%pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2" "langchain-elasticsearch<0.3" "jq==1.12.0"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Note to Meyself: Purpose of This Cell
The purpose of this cell is to install all the libraries that will be used throughout the project. The required components for building our RAG (Retrieval-Augmented Generation) system include:

LangChain
OpenAI
Elasticsearch
jq
Text Splitters
Retrievers
This cell is only responsible for setting up the environment.

At this stage:

No AI model is being executed.
No connection to Elasticsearch is established.
No data is loaded or processed.
Its sole purpose is to ensure that all necessary dependencies are installed and ready for use in the following steps.

In [3]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

# Note to Meyself: Purpose of This Cell
The purpose of this cell is to load the classes and modules that will be used throughout the project into memory.

At this stage, the notebook is not:

Connecting to OpenAI
Connecting to Elasticsearch
Asking any questions
Generating embeddings
This cell only imports the necessary tools and dependencies required for building the RAG system.

In Python, a successful import operation usually produces no output. Therefore, seeing an empty output after running this cell is completely normal and indicates that the imports were loaded successfully.

## Connect to Elasticsearch

In this lab, you'll use Elasticsearch as the vector database and search engine that stores your documents and retrieves the most relevant ones in response to user queries.

**What is Elasticsearch?**

Elasticsearch is a distributed search and analytics engine designed for fast retrieval over large collections of data. In AI applications, it can store both text and vector embeddings, making it well suited for Retrieval-Augmented Generation (RAG) workflows where an LLM retrieves relevant context before generating a response.


**Getting started with Elastic Cloud:**

1. If you don't already have one, sign up for a free trial: [https://www.elastic.co/](https://www.elastic.co/).
2. During sign-up (or afterwards), create a deployment/project. Either a classic deployment or a serverless project will work for this lab.
3. Once your deployment/project is ready, you'll need to find your **Cloud ID** and create an **API key**. 

<br>

> To find your **Cloud ID** and create an **API key**, 
> follow the instructions on the link below.
> 
>   👇👇👇
>
> https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
>
<br>

# Note to My Self: Purpose of The Following Code Cell
The purpose of this cell is not to write code yet, but to prepare the required accounts and credentials that will be used later in the project.

Do not run this cell yet. Before executing it, you need to obtain the following information:

Elastic Cloud ID
Elastic API Key
OpenAI API Key
These credentials are required to establish secure connections to the external services used in the RAG system.

Without them, the application will not be able to:

Connect to Elasticsearch
Connect to OpenAI
Generate embeddings
Perform retrieval and question-answering tasks
Once all three credentials are available, I can safely run the cell and proceed with the project setup! 

In [20]:
#
#
# To find your Cloud ID and create an API key, 
# follow the instructions on the link below:
#
# 👇👇👇
#
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
#
#
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

index_name = "ironhack-multi-query-retriever"

# Note to my self: 
What each line means?

ELASTIC_CLOUD_ID = ... asks for the Elastic Cloud ID and stores it in a variable.
ELASTIC_API_KEY = ... asks for the Elastic API key and stores it.
OPENAI_API_KEY = ... asks for the OpenAI API key and stores it.
getpass("...") displays a prompt; the text I type is hidden.
= assigns the value on the right to the variable name on the left.
embeddings = OpenAIEmbeddings(...) prepares the OpenAI embedding tool using your OpenAI key. The parentheses call the tool’s setup, and openai_api_key=OPENAI_API_KEY passes the key stored in that variable.
index_name = "..." sets the name the lab will use for its Elasticsearch index. The quotation marks mean this value is text.


## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [15]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [16]:
from langchain_community.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    metadata.update(
        {
            key: record.get(key)
            for key in ("name", "summary", "url", "category", "updated_at")
        }
    )

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [23]:
vectorstore = ElasticsearchStore.from_documents(
    docs,
    embeddings,
    index_name=index_name,
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

In [22]:
cloud_id_check = ELASTIC_CLOUD_ID.strip().split(":", 1)
print("Has name and encoded section:", len(cloud_id_check) == 2 and bool(cloud_id_check[0]) and bool(cloud_id_check[1]))
print("Has extra spaces at either end:", ELASTIC_CLOUD_ID != ELASTIC_CLOUD_ID.strip())

Has name and encoded section: True
Has extra spaces at either end: False


# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [27]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['What does the NASA sales team do and what are their responsibilities?  ', 'Can you provide information about the structure and functions of the NASA sales team?  ', 'Who are the members of the NASA sales team and what roles do they play?']


---- Answer ----
content='The NASA sales team refers to the sales organization that operates within the North America and South America region, which is abbreviated as NASA. This region encompasses the United States, Canada, Mexico, as well as Central and South America. The NASA sales team is structured to effectively serve customers in these areas and achieve business objectives.\n\nThe leadership of the NASA sales team is divided between two Area Vice-Presidents: Laura Martinez oversees the North America segment, while Gary Johnson is responsible for the South America segment. Each regional sales team, including the NASA team, consists of dedicated account managers, sales representatives, and support staff. Their primary responsibilities include identifying and pursuing new business opportunities, nurturing existing client relationships, and ensuring customer satisfaction.\n\nThe NASA sales team collaborates closely with other departments, such as marketing, product development, and 

In [26]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=OPENAI_API_KEY,
)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

<br>

## 🚀 Your turn

Time to make this lab your own!


So far we've asked a single question ("what is the nasa sales team?") and looked at one answer. Now it's your turn to try the pipeline with your own questions and see how `MultiQueryRetriever` handles them.


Here's what to do:

1. **Pick 2 new questions** — Look through the sample dataset (or just get creative) and write down 2 questions you'd like the chatbot to answer, different from the example above.
2. **Ask each question** — For each question, copy the last code cell, replace the question passed to `chain.invoke(...)`, and run it.
3. **Check the logs** — Because logging is enabled, you'll see the alternative questions `MultiQueryRetriever` generated for you in the cell output. Take a look at them.
4. **Reflect** — In a markdown cell, briefly note: Did the generated questions look useful? Was the final answer correct and well grounded in the retrieved context, or did the model seem to be missing information?

<br>

💡 **Tip:**

- If an answer looks off, try asking the same question in a different way, or peek at the generated queries to see if they drifted from what you actually meant.

<br>

⭐️ **Bonus ideas:**

- Change `retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)` to instead use `vectorstore.as_retriever()` directly (no multi-query), and compare the answers you get for the same questions. Does `MultiQueryRetriever` actually help?
- Look at the `k` parameter of `as_retriever()` (e.g. `vectorstore.as_retriever(search_kwargs={"k": 3})`) to control how many passages are retrieved per query, and see how that affects the final answer.
- Edit `LLM_CONTEXT_PROMPT` to change the tone or format of the final answer (e.g. ask for a short bullet-point summary instead of a verbose answer), and see how that changes the output.
- Ask a question that isn't covered by the sample dataset at all — does the chatbot correctly say it doesn't know, or does it make something up?

In [28]:
question_1 = "Which sales regions does the company have, and who leads the Europe region?"
answer_1 = chain.invoke(question_1)

print("---- Question 1 ----")
print(question_1)
print("---- Answer ----")
print(answer_1)

INFO:langchain.retrievers.multi_query:Generated queries: ['What are the different sales regions established by the company, and who is the leader for the European region?', "Can you provide a list of the company's sales regions and identify the person in charge of the Europe region?", 'What sales territories does the company operate in, and who oversees the sales efforts in Europe?']


---- Question 1 ----
Which sales regions does the company have, and who leads the Europe region?
---- Answer ----
content='The company has four main sales regions: \n\n1. **The Americas** - This region includes the United States, Canada, Mexico, and Central and South America.\n2. **Europe** - This region covers the entire continent, including countries such as the United Kingdom, Germany, France, Spain, Italy, and others.\n3. **Asia-Pacific** - This region encompasses countries like China, Japan, South Korea, India, Australia, and New Zealand.\n4. **Middle East & Africa** - This region includes countries across the Middle East and Africa, such as the United Arab Emirates, Saudi Arabia, South Africa, and Nigeria.\n\nThe Europe region is led by **Rajesh Patel**, who serves as the Area Vice-President for that region. Each regional sales team is composed of dedicated account managers, sales representatives, and support staff, all working under their respective Area Vice-Presidents to achie

In [29]:
question_2 = "According to the May 2023 hybrid work update, how many days per week should employees work in the office and remotely?"
answer_2 = chain.invoke(question_2)

print("---- Question 2 ----")
print(question_2)
print("---- Answer ----")
print(answer_2)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the recommended office and remote work days for employees as outlined in the May 2023 hybrid work update?  ', '2. How many days each week are employees expected to work in the office versus remotely based on the May 2023 hybrid work guidelines?  ', '3. Can you provide details on the weekly in-office and remote work schedule for employees from the May 2023 hybrid work update?']


---- Question 2 ----
According to the May 2023 hybrid work update, how many days per week should employees work in the office and remotely?
---- Answer ----
content="According to the May 2023 hybrid work update, employees are required to work from the office three days a week, with two days designated for remote work. This adjustment reflects the company's ongoing commitment to employee well-being while also promoting a structured work environment. Employees are encouraged to communicate with their supervisors and the HR department to establish their specific in-office workdays." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 236, 'total_tokens': 308, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ce

### Reflection

The two questions target distinct sample documents: `Sales Organization Overview` and `Wfh Policy Update May 2023`. The latter is dated May 1, 2023, and states that employees work three days in the office and two remotely. The sales-organization document contains the regional structure and leadership information requested by the first question.

After running both question cells, compare the generated query logs with each original question and check the retrieved source passages against the answers. The query variations and live answers have not been evaluated here because running the chain requires working Elastic Cloud and OpenAI credentials.

# MY TAKE ON IT ALL: 

The generated queries were useful because they rephrased my questions while keeping the same topics. For the hybrid-work question, the answer said employees should work three days in the office and two days remotely, which matched the policy information in the notebook. The answer also added advice about communicating with supervisors and HR; I would check the retrieved source before treating that extra detail as confirmed. Overall, this exercise showed me how MultiQueryRetriever searches using different phrasings, while the final answer still needs to be checked against its sources which I think might be an issue to do manuelly all the time. I am sure there is another way to automize the final checks too. I have to dig into it more. 
